In [9]:
import pandas as pd

In [36]:
def filter_format_taxonomy(level, filepath=None, df=None):
    """
    Preprocessing the files to form into counts df

    :param filepath: path to the taxonomy counts
    :param type: str
    :param df: pandas dataframe in same format of taxonomy counts
    :param type: panda series
    :param level: taxonomic level to count to, must match case/spelling in the df or tsv
    :param type: str
    """
    # if user inputs file
    if df is not None:
        df = filter_file(df, level)
    # filepath may be specified instead of df
    elif filepath is not None:
        df = pd.read_csv(filepath, sep="\t")
        # read in the csv
        df = filter_file(df, level)
        return df
    else:
        raise Exception("Need to input a file or dataframe")

In [30]:
def filter_file(df, level):
    """
    Preprocessing by filtering out species without taxonomic ids

    :param df path to the taxonomy counts
    :param type: str
    :param level: taxonomic level to count to, must match case/spelling in the df or tsv
    :param type: str
    :return: df at certain taxonomic level with count of cluster
    :return type: pd Dataframe
    """
    # the df counts at each level so no need to condense counts
    df = df.loc[df["level"] == level]
    df = df[["cluster_ID", "taxon", "count"]]
    # removing any column with a NA
    df = df.loc[~df.isnull().any(axis=1)]
    # pivot to get the classic df format
    df = df.pivot(index="cluster_ID", columns="taxon", values="count")
    df = df.fillna(0)
    return df

In [5]:
def AA_count_file(df, level, outpath):
    """
    Making a taxonomic counts file with Nomburg's clustering
    :param df path to the taxonomy counts
    :param type: str
    :param level: taxonomic level to count to, must match case/spelling in the df or tsv
    :param type: str
    :param outpath: the file to write the taxonomic counts dataframe
    :param type: str
    """
    df = pd.read_csv(df, sep="\t")
    # select the taxonomic level and the subcluster rep
    # subcluster rep is the species rep
    df = df.loc[:, [level, "subcluster_rep"]]
    # remove null, if there is NA for taxonomic naming remove
    df = df.loc[~df.isnull().any(axis=1)]
    # some species contain the same protein cluster
    # (or they are different taxonomic id/strains)
    df = df.drop_duplicates()
    df["count"] = 1
    # rename column for compatibility
    df = df.rename(columns={"subcluster_rep": "cluster_ID"})
    # pivot the count to 1
    df = df.pivot(index="cluster_ID", columns=level, values="count")
    df = df.fillna(0)
    # write df to specified outpath
    df.to_csv(outpath)

In [ ]:
def make_taxon_from_amino_acids(mmseqs_clusters, tax_prot):
    """
    Taking the mmseqs clustering and creating a dataframe of the counts of
    representatives in every virus
    :param mmseqs_clusters: file of the sequence member (0th) and representative (1st)
    :param type: str
    :param species_list: filepath to tsv of protein (0th) and corresponding species
    :param type: str
    """
    # loading prot, species df
    species_prot = pd.read_csv(tax_prot, sep="\t", names=["mem", "species"])
    # rep, mem cluster csv loading here
    mmseqs = pd.read_csv(mmseqs_clusters, sep="\t", names=["rep", "mem"])
    # merge the prot from species to mem
    prot_master = pd.merge(species_prot, mmseqs, on="mem", how="inner")
    # add a one column
    prot_master["count"] = 1
    # remove the mems column
    prot_master = prot_master.drop("mem", axis=1)
    # dropping duplicates, it makes sense that distinct protein
    # from species with the same rep.
    prot_master = prot_master.drop_duplicates()
    # rename column to cluster id
    prot_master = prot_master.rename(columns={"rep": "cluster_ID"})
    prot_master = prot_master.pivot(
        index="cluster_ID", columns="species", values="count"
    ).fillna(0)
    # index is included in default settings
    prot_master.to_csv("../results/amino_acid_count_df.csv")


In [7]:
def filter_for_knn(df, species_list, outpath):
    """
    Filtering df for species that are captured with the amino acid & structural clusts.
    :param df: path to df column- species, row- cluster ID, value binary indicator
    :param type: pd Datadrame
    :param species_list: filepath of csv of shared species
    :param type: str
    :param outpath: path to write the filtered species df
    :param type: str
    """
    # read in the csv
    df = pd.read_csv(df)
    # open the csv to read
    with open(species_list, "r") as f:
        # read the one line, split the csv into a list
        species = f.readline().strip().split(",")
        # column retrieval of the species
        filtered_df = df[species]
        filtered_df.to_csv(outpath)

In [25]:
def main():
# path to the file
    #file=sys.argv[1]
    #outpath=sys.argv[2]

    # THESE LINES WERE FOR STRUCUTRAL CLUSTERS
    file="../code_and_intermediate_data/intermediate_data/merged_clusters.counts.tsv"
    outpath="../outputs/count_df/species_cluster_count.csv"
    filter_format_taxonomy("species",outpath,filepath=file)

    # THESE LINES ARE FOR THE AMINO ACID CLUSTERS FROM MMSEQS
    species="../proteins/nomburg_prot_modified_species.tsv"
    clusters="../outputs/amino_acid_clusters/viral_prots_cluster.tsv"
    make_taxon_from_amino_acids(clusters, species)

    # THESE LINES ARE FOR THE AMINO ACID CLUSTERS FROM NOMBURG'S FILE
    file="../code_and_intermediate_data/intermediate_data/merged_clusters.tax.tsv"
    outpath="../outputs/count_df/nomburg_amino_acid.csv"
    AA_count_file(file,"species",outpath)
    
    ## THESE LINES ARE FOR FILTERING THE SPECIES FOR THE ONES SHARED BETWEEN AA AND STR
    amino_acid_df="../outputs/count_df/amino_acid_cluster.csv"
    #structure_df="../outputs/count_df/species_structure_cluster_count.csv"
    species="../outputs/knn_inputs/header_both.csv"
    outdir_aa="../outputs/knn_inputs/amino_acid_knn_df.csv"
    filter_for_knn(amino_acid_df, species, outdir_aa)

In [ ]:
def main():
    ## Shared protein fold df construction
    file="../intermediate_files/nomburg_files/merged_clusters.counts.tsv"
    outpath="../results/structure_df.csv"
    str_df=filter_format_taxonomy("species",filepath=file)
    str_df.to_csv(outpath)
    ## Shared protein sequence df construction 
    species="../intermediate_files/nomburg_prot_modified_species.tsv"
    clusters="../intermediate_files/viral_prots_cluster.tsv"
    make_taxon_from_amino_acids(clusters, species)
    ## Filter for proteins found in both
    structure_df="../results/structure_df.csv"
    amino_acid_df="../results/amino_acid_count_df.csv"
    #structure_df="../outputs/count_df/species_structure_cluster_count.csv"
    species="../intermediate_files/header_both.csv"
    outdir_aa="../results/knn_amino_df.csv"
    outdir_str="../results/knn_str_df.csv"
    filter_for_knn(amino_acid_df, species, outdir_aa)
    filter_for_knn(structure_df, species, outdir_str)

In [45]:
main()